# Extracting Features

Once we have extracted scan segments, we treat these as individual observations. Using these observations, we want to construct features from the variable-length time-series data.

We produce datasets that can vary along the following two axes:
1. Simple Statistical Features (mean, range, max, min, stddev) OR Wavelet Features
2. Resolution (we can "chop up" segments of a certain length into smaller pieces)

The segment's lengths can be calculated as the sum of deltas, which are helpfully provided in the csvs folder. The target variable is ID2, which tells us the severity of the overhang on a scale of 0-10.



In [1]:
import os

import numpy as np
import pandas as pd
import tqdm as tqdm
import pywt

In [ ]:
def simple_statistics(time_series):
    # time_series is a numpy array
    # return the mean, max, min, stddev
    return {
        'mean': np.mean(time_series),
        'max': np.max(time_series),
        'min': np.min(time_series),
        'stddev': np.std(time_series)
    }

def wavelet_features(signal, wavelet='db4', level=5):
    coeffs = pywt.wavedec(signal, wavelet, level=min(level, pywt.dwt_max_level(len(signal), wavelet)))
    # Energy in each sub-band (approximation + details)
    energies = [np.sum(c**2) / len(c) for c in coeffs]
    total = sum(energies)
    return {f'wavelet_level_{i}_energy_ratio': e/total for i, e in enumerate(energies)}


In [ ]:
# these are in units of cm, for how long each segment is allowed to be
# if a segment is longer than resolution, then it get divided in equal pieces
# until each piece is below resolution
# if None, then don't divide
resolutions = [None, 4, 2, 1]

In [ ]:
data_dir = "./csvs/"
new_data_dir = "./new_csvs/"
os.makedirs(new_data_dir, exist_ok=True)

files = os.listdir(data_dir)
files = [f for f in files if f.endswith(".csv")]
files.sort(key=lambda x: int(x.split(".")[0][5:]))
do_this_many = len(files) # NOTE: can be less if you want to test

for feature_function in [simple_statistics, wavelet_features]:
    for resolution in resolutions:
        # parse all files and prepare to put them in a single DataFrame
        df_list = [] # will contain a list of dictionaries
        for f_idx in tqdm(range(do_this_many)):
            file_path = os.path.join(data_dir, files[f_idx])
            df = pd.read_csv(file_path)
            
        # create a DataFrame from the list of dictionaries
        combined_df = pd.DataFrame(df_list)
        # save the DataFrame to a new CSV file
        combined_df.to_csv(os.path.join(new_data_dir, f"{feature_function.__name__}_resolution_{resolution}.csv"), index=False)